# V3 / 00 — UAV input inventory and spatial data audit

**Start here.** CPU-only, Google Colab + Drive. This notebook discovers the authoritative input files in `Musanga/Final/Data`, inventories native RGB orthomosaics, DSM, ROI and tree annotations, and verifies their spatial relationships. It does **not** rewrite, reproject, or copy input data.

**V3 data contract:** `Final/Data/Ortho_RGB/` holds six native RGB orthomosaics and the modelling ROI shapefile (or ROI may be in the `Final/Data/` root); `Final/Data/DSM/` currently contains only Baego DSM; `Final/Data/Annotations/` holds tree-annotation shapefiles. **No Luki**. MOS is **one site with three distinct mosaics** and **one shared crown annotation inventory**. Report these separately.

Outputs: small audit CSV and JSON files in `Final/Outputs/V3/00_input_audit/`. No training or probability inference in Notebook 00. Assumptions about labels (for example, `Class_ID==1`) are **inspected, not silently imposed**; modelling class mapping is a later V3 decision.

**New exhaustive labels:** `Annotations/Circles_Annotations.shp` identifies Musanga-positive polygon areas within `Annotations/Circles_ROI.shp` boundaries. Unannotated area *inside those circle ROIs* is confirmed non-Musanga; everything outside is unlabelled. This inventory is distinct from per-site, partially annotated tree crowns.

**Authoritative V3 spatial mask:** the general `ROI.shp` delineates where RGB is usable and all predictions must be made. Pixels outside this ROI are not sampled, trained on or predicted. Exhaustive `Circles_ROI.shp` is a distinct ground-truth annotation mask.


## 01 — Environment and Drive setup

Mount Google Drive, locate the authoritative `Final/Data` folders, and establish the audit output location. Original data are never modified.


In [ ]:
# V3 Notebook 00 / Block 01 — Mount Drive and locate the FINAL data directory
%pip -q install rasterio geopandas pyogrio shapely pandas matplotlib
from pathlib import Path
import json, re, math, warnings
import numpy as np, pandas as pd, geopandas as gpd, rasterio, matplotlib.pyplot as plt
from shapely.geometry import box
from shapely.validation import explain_validity
from IPython.display import display
from google.colab import drive

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
MYDRIVE=Path("/content/drive/MyDrive")

# Set this manually only if your MyDrive location is not among these two:
DATA_ROOT_OVERRIDE=None
CANDIDATE_DATA_ROOTS=[
    MYDRIVE/"Colab Notebooks/Musanga/Final/Data",
    MYDRIVE/"Musanga/Final/Data",
]
DATA=Path(DATA_ROOT_OVERRIDE) if DATA_ROOT_OVERRIDE else next(
    (p for p in CANDIDATE_DATA_ROOTS if p.is_dir()),None)
assert DATA is not None,(
    "Cannot find Final/Data. Set DATA_ROOT_OVERRIDE above to the actual "
    "Google Drive path. Tried: "+", ".join(map(str,CANDIDATE_DATA_ROOTS)))
RGB_DIR=DATA/"Ortho_RGB"
DSM_DIR=DATA/"DSM"
ANN_DIR=DATA/"Annotations"
for d in (RGB_DIR,DSM_DIR,ANN_DIR):
    assert d.is_dir(),f"Required folder missing: {d}"
OUT=DATA.parent/"Outputs"/"V3"/"00_input_audit"
OUT.mkdir(parents=True,exist_ok=True)
print("Authoritative data directory:",DATA)
print("Audit reports (only writes here):",OUT)
print("Input subfolders:")
for d in (RGB_DIR,DSM_DIR,ANN_DIR):
    print(" ",d.name,":",len(list(d.iterdir())),"entries")


## 02 — Native RGB orthomosaics and DSM inventory

Identify six orthomosaics across four site groups, including the three MOS mosaics, and inspect native resolution, CRS, band structure and DSM availability.


In [ ]:
# V3 Notebook 00 / Block 02 — Native image registry (no imagery is loaded)
def tifs(folder):
    return sorted([p for p in folder.iterdir() if p.is_file()
                   and p.suffix.lower() in (".tif",".tiff")],
                  key=lambda p:p.name.lower())
rgb_paths=tifs(RGB_DIR)
dsm_paths=tifs(DSM_DIR)
print("RGB mosaics found:",len(rgb_paths))
for p in rgb_paths:print("  ",p.name)
print("DSM files found:",len(dsm_paths))
for p in dsm_paths:print("  ",p.name)

# Matching uses filenames only to assign site; actual basenames are NOT assumed.
# No Luki in V3: fail if unexpected files are included.
def assign_site(name):
    x=name.lower()
    if "luki" in x:return "Luki"
    if "baeg" in x:return "Baego"
    if "yangambi" in x:return "Yangambi"
    if "yoko" in x:return "Yoko"
    if re.search(r"(^|[^a-z])mos([_\-.]|$)",x):return "MOS"
    return "UNRECOGNIZED"
def mos_group(name):
    m=re.search(r"(?<!\d)([135])[_-]([246])(?!\d)",name.lower())
    return f"MOS_{m.group(1)}_{m.group(2)}" if m else None

entries=[];raster_bounds={}
for kind,paths in (("RGB",rgb_paths),("DSM",dsm_paths)):
    for p in paths:
        site=assign_site(p.stem)
        try:
            with rasterio.open(p) as src:
                r=dict(role=kind,site=site,mos_group=mos_group(p.stem) if site=="MOS" else "",
                       filename=p.name,path=str(p),size_GB=round(p.stat().st_size/1e9,3),
                       width=src.width,height=src.height,bands=src.count,
                       dtype_first3="/".join(src.dtypes[:3]),
                       crs=str(src.crs),pixel_x=abs(src.res[0]),pixel_y=abs(src.res[1]),
                       pixel_area_m2=abs(src.transform.a*src.transform.e-src.transform.b*src.transform.d),
                       nodata=str(src.nodata),
                       bounds=str(tuple(round(v,2) for v in src.bounds)),
                       colorinterp="/".join(str(v) for v in src.colorinterp[:3]),
                       error="")
                raster_bounds[p.name]=(src.crs,box(*src.bounds))
                if not src.crs: r["error"]="Missing CRS"
                elif kind=="RGB" and src.count<3:r["error"]="RGB requires at least 3 bands"
                elif kind=="RGB" and src.count>3:
                    print("NOTE:",p.name,"has",src.count,"bands; V3 RGB uses first 3 only")
        except Exception as exc:
            r=dict(role=kind,site=site,filename=p.name,path=str(p),
                   error=f"{type(exc).__name__}: {exc}")
        entries.append(r)
rasters=pd.DataFrame(entries)
display(rasters)
rgb=rasters[rasters.role=="RGB"]
print("RGB mosaic counts per site:",rgb.site.value_counts().to_dict())
assert len(rgb_paths)==6, f"Expected 6 RGB orthomosaics; found {len(rgb_paths)}"
assert rgb.site.value_counts().to_dict()=={"MOS":3,"Baego":1,"Yangambi":1,"Yoko":1},(
    "Unexpected site distribution or naming; inspect the table and update assign_site.")
assert not rasters.error.fillna("").astype(str).str.len().gt(0).any(), "Raster metadata errors in table"
if len(dsm_paths)!=1 or assign_site(dsm_paths[0].stem)!="Baego":
    print("NOTE: DSM availability differs from expected single Baego DSM.")
rasters.to_csv(OUT/"00_raster_inventory.csv",index=False)


## 03 — Crown annotations and general modelling ROI

Inventory the partially annotated tree-crown files, check label attributes and geometry integrity, and locate **`ROI.shp`** (the general modelling area, not the exhaustive-circle boundary file).


In [ ]:
# V3 Notebook 00 / Block 03 — Annotation registry, label QA and ROI discovery
# Annotation filenames may vary (plural Tree_Annotations or singular Tree_Annotation).
vector_exts={".shp",".gpkg",".geojson"}
ann_files=sorted([p for p in ANN_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in vector_exts],key=lambda p:p.name.lower())
print("Annotation vector files:",len(ann_files))
for p in ann_files:print(" ",p.relative_to(ANN_DIR))

# Prefer ROI.shp within Ortho_RGB or directly under Final/Data. Search only under Final/Data.
roi_candidates=sorted([p for p in DATA.rglob("*") if p.is_file()
                       and p.suffix.lower() in vector_exts
                       and (p.stem.lower()=="roi" or p.stem.lower().startswith("roi_"))],
                      key=lambda p:(len(p.parts),str(p)))
print("ROI candidates:",[str(p.relative_to(DATA)) for p in roi_candidates])
assert len(roi_candidates)==1, (
    "Exactly one ROI vector is expected. Set an explicit ROI path if multiple candidates appear.")
ROI_FILE=roi_candidates[0]
def inspect_vector(path,role):
    records=[]
    try:
        layers=[None]
        if path.suffix.lower()==".gpkg":
            import fiona
            layers=fiona.listlayers(path)
        for layer in layers:
            g=gpd.read_file(path,layer=layer) if layer is not None else gpd.read_file(path)
            bad=int((~g.geometry.is_valid).sum())
            missing=int(g.geometry.isna().sum())
            record=dict(role=role,site=assign_site(path.stem),
                 file=path.name,path=str(path),layer=layer or "",
                 n_features=len(g),crs=str(g.crs),geom_types=";".join(sorted(map(str,g.geom_type.unique()))),
                 invalid_geometries=bad,null_geometries=missing,
                 fields=";".join(map(str,g.columns)),
                 label_field="Class_ID" if "Class_ID" in g.columns else
                    ("Espece" if "Espece" in g.columns else
                     ("Nom" if "Nom" in g.columns else "")),
                 error="")
            records.append(record)
    except Exception as exc:
        records.append(dict(role=role,site=assign_site(path.stem),file=path.name,
                            path=str(path),error=f"{type(exc).__name__}: {exc}"))
    return records

ann_records=[]
for f in ann_files:ann_records+=inspect_vector(f,"tree_annotations")
roi_records=inspect_vector(ROI_FILE,"ROI")
ann_table=pd.DataFrame(ann_records)
display(ann_table)
display(pd.DataFrame(roi_records))
assert not any(r["error"] for r in ann_records+roi_records), "Vector read failures: inspect table"
ann_table.to_csv(OUT/"00_annotation_inventory.csv",index=False)
pd.DataFrame(roi_records).to_csv(OUT/"00_roi_inventory.csv",index=False)

# Class frequencies are descriptive; do not assume all datasets use identical coding.
freq=[]
for f in ann_files:
    g=gpd.read_file(f)
    field=next((k for k in ("Class_ID","Espece","Nom") if k in g.columns),None)
    if field is not None:
        for value,count in g[field].fillna("<MISSING>").astype(str).value_counts().items():
            freq.append(dict(file=f.name,site=assign_site(f.stem),
                             label_field=field,value=value,count=int(count)))
if freq:
    display(pd.DataFrame(freq))
    pd.DataFrame(freq).to_csv(OUT/"00_annotation_label_counts.csv",index=False)

# Check mandatory shapefile sidecars; absence of .prj means geographic alignment is unknown.
missing_sidecars=[]
for f in [ROI_FILE]+[x for x in ann_files if x.suffix.lower()==".shp"]:
    for suffix in (".shx",".dbf",".prj"):
        if not f.with_suffix(suffix).is_file():
            missing_sidecars.append(str(f.with_suffix(suffix)))
print("Missing shapefile sidecars:",missing_sidecars or "none")
assert not missing_sidecars,"Fix incomplete shapefile sets before modelling."


## 04 — Spatial consistency and shared MOS annotations

Check alignment between each orthomosaic, its per-site crown annotations and the general ROI. The three MOS mosaics share one annotation dataset.


In [ ]:
# V3 Notebook 00 / Block 04 — Associate MOS mosaics with ONE shared annotation file
# Compare vector and raster footprint intersections in the native raster CRS.
from shapely.ops import unary_union
ann_site={}
for site in ("Baego","Yangambi","Yoko","MOS"):
    matches=[p for p in ann_files if assign_site(p.stem)==site
             and "tree" in p.stem.lower()]
    print(site,": annotation matches:",[p.name for p in matches])
    assert len(matches)==1,(
        f"Expected one tree-annotation inventory for {site}; found {len(matches)}. "
        "Inspect names or define an explicit mapping.")
    ann_site[site]=matches[0]
roi_g=gpd.read_file(ROI_FILE)
assert roi_g.crs is not None,"ROI has no CRS"
roi_valid=roi_g.geometry.dropna()
assert len(roi_valid) and roi_valid.is_valid.all(),"ROI has missing/invalid geometry"
roi_geom=unary_union(roi_valid.to_list())
print("ROI:",ROI_FILE.name,"features",len(roi_g),"CRS",roi_g.crs)

spatial=[]
for p in rgb_paths:
    site=assign_site(p.stem)
    with rasterio.open(p) as src:
        footprint=box(*src.bounds)
        annotations=gpd.read_file(ann_site[site])
        assert annotations.crs is not None,f"Missing annotation CRS: {ann_site[site].name}"
        ann_aligned=annotations.to_crs(src.crs)
        # Only bounds-test per feature, not an expensive polygon union.
        intersects=ann_aligned.geometry.intersects(footprint)
        roi_aligned=roi_g.to_crs(src.crs)
        roi_intersect=unary_union(roi_aligned.geometry.to_list()).intersection(footprint).area
        footprint_area=footprint.area
        spatial.append(dict(site=site,mosaic=p.name,
            annotation_file=ann_site[site].name,
            annotation_total=len(annotations),
            annotation_intersecting_mosaic=int(intersects.sum()),
            annotation_outside_mosaic=int((~intersects).sum()),
            roi_mosaic_intersection_m2=roi_intersect,
            roi_fraction_of_mosaic_bbox=roi_intersect/max(footprint_area,1),
            imagery_crs=str(src.crs),annotations_crs=str(annotations.crs),
            roi_crs=str(roi_g.crs)))
relations=pd.DataFrame(spatial)
display(relations)
relations.to_csv(OUT/"00_mosaic_annotation_roi_relationships.csv",index=False)
print("MOS shared annotation file:",ann_site["MOS"].name)
print("MOS annotation records are NOT duplicated into three source files.")
if (relations.annotation_intersecting_mosaic==0).any():
    warnings.warn("One or more mosaic/annotation pairs have no spatial intersection. "
                  "Check naming, coordinate systems and annotation coverage.")
if (relations.roi_mosaic_intersection_m2==0).any():
    warnings.warn("One or more mosaics have no ROI overlap; inspect ROI CRS and location.")

# Lightweight overview figure from geometry, no large UAV pixel reads.
fig,ax=plt.subplots(figsize=(9,8))
for _,r in relations.iterrows():
    crs,geom=raster_bounds[r.mosaic]
    display_geom=gpd.GeoSeries([geom],crs=crs).to_crs(roi_g.crs).iloc[0]
    x,y=display_geom.exterior.xy
    ax.plot(x,y,label=r.site+(" "+str(mos_group(r.mosaic)) if r.site=="MOS" else ""))
gpd.GeoSeries([roi_geom],crs=roi_g.crs).boundary.plot(ax=ax,color="black",linewidth=2)
ax.set_title("RGB mosaic bounding boxes and model ROI (same CRS)")
ax.set_aspect("equal");ax.legend(fontsize=8)
fig.savefig(OUT/"00_mosaic_roi_overview.png",dpi=150,bbox_inches="tight")
plt.show()


## 05 — Native RGB illumination **inside the modelling ROI only**

The general `ROI.shp` defines the valid UAV-data footprint, for **both training and model predictions**. Sample 256-pixel windows centered within its intersection with each orthomosaic; compute statistics **only from pixels inside the ROI**. Do not sample or model outside the ROI. `Circles_ROI.shp` separately defines exhaustive annotation support and does not replace the general ROI.

Report valid-window counts and sampling acceptance, especially for Baego. All imagery and vector inputs remain unchanged.

When the general ROI does not overlap a mosaic, record `NO_ROI_OVERLAP` and continue. The mosaic is excluded from both illumination and future modelling until ROI coverage is resolved.


In [ ]:
# Block 05 — ROI-CONSTRAINED illumination audit (CPU; no inference)
# The GENERAL ROI.shp is the authoritative valid image/model footprint.
# Circles_ROI.shp has a DIFFERENT role: exhaustive annotation regions only.
from rasterio.windows import Window, from_bounds
from rasterio.features import geometry_mask
from rasterio.transform import from_bounds as transform_from_bounds
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.prepared import prep

LIGHT_WINDOWS00=64              # target valid windows PER RGB mosaic
LIGHT_PIXELS00=1000             # max sampled pixels per accepted window
LIGHT_WINDOW_PX00=256
LIGHT_MIN_PIXELS00=100          # valid RGB+ROI pixels in a window
LIGHT_SEED00=20261015
LIGHT_MAX_TRIES00=1000          # bounded to avoid lengthy sampling
LIGHT_CANDIDATE_GRID00=400     # coarse ROI candidate raster, no RGB read

illum_rows00=[];illum_pixels00=[];illum_attempts00=[]
for i,p in enumerate(rgb_paths):
    rng=np.random.default_rng(LIGHT_SEED00+i)
    with rasterio.open(p) as ds:
        assert ds.crs is not None and ds.count>=3,(p.name,ds.crs,ds.count)
        # Reproject the GENERAL ROI to this native mosaic's CRS, in memory only.
        roi_this00=roi_g.to_crs(ds.crs)
        eligible_geom00=unary_union(roi_this00.geometry.to_list()).intersection(box(*ds.bounds))
        if eligible_geom00.is_empty or eligible_geom00.area <= 0:
            warnings.warn(
                f"{p.name}: NO general ROI.shp overlap. Skipping. "
                "Do not model this mosaic until the ROI is checked."
            )
            illum_attempts00.append(dict(
                site=assign_site(p.stem),mosaic=p.name,
                accepted_windows=0,candidates_tried=0,
                acceptance_pct=np.nan,roi_candidate_cells=0,
                roi_overlap_area_m2=0.0,status="NO_ROI_OVERLAP"))
            continue
        # Rasterize ONLY the ROI to a coarse candidate grid, not the huge UAV mosaic.
        left,bottom,right,top=eligible_geom00.bounds
        ratio=(right-left)/max(top-bottom,1e-9)
        grid_w=max(1,min(LIGHT_CANDIDATE_GRID00,int(round(LIGHT_CANDIDATE_GRID00*ratio))))
        grid_h=max(1,min(LIGHT_CANDIDATE_GRID00,int(round(LIGHT_CANDIDATE_GRID00/max(ratio,1e-9)))))
        grid_transform00=transform_from_bounds(left,bottom,right,top,grid_w,grid_h)
        coarse_roi00=geometry_mask([eligible_geom00],out_shape=(grid_h,grid_w),
                                   transform=grid_transform00,invert=True)
        candidate_y,candidate_x=np.nonzero(coarse_roi00)
        if len(candidate_x)==0:
            warnings.warn(f"{p.name}: ROI overlap missed by coarse candidate grid.")
            illum_attempts00.append(dict(
                site=assign_site(p.stem),mosaic=p.name,
                accepted_windows=0,candidates_tried=0,
                acceptance_pct=np.nan,roi_candidate_cells=0,
                roi_overlap_area_m2=float(eligible_geom00.area),
                status="ROI_TOO_SMALL_FOR_GRID"))
            continue
        accepted00=0;tries00=0
        while accepted00<LIGHT_WINDOWS00 and tries00<LIGHT_MAX_TRIES00:
            tries00+=1
            c=int(rng.integers(0,len(candidate_x)))
            # Center candidate within an eligible coarse-grid cell.
            gx=float(candidate_x[c])+float(rng.random())
            gy=float(candidate_y[c])+float(rng.random())
            mx,my=grid_transform00*(gx,gy)
            rr,cc=ds.index(mx,my)
            row=max(0,min(ds.height-LIGHT_WINDOW_PX00,rr-LIGHT_WINDOW_PX00//2))
            col=max(0,min(ds.width-LIGHT_WINDOW_PX00,cc-LIGHT_WINDOW_PX00//2))
            win=Window(col,row,min(LIGHT_WINDOW_PX00,ds.width-col),
                       min(LIGHT_WINDOW_PX00,ds.height-row))
            if win.width<=0 or win.height<=0:continue
            roi_pixels00=geometry_mask(
                [eligible_geom00],out_shape=(int(win.height),int(win.width)),
                transform=ds.window_transform(win),invert=True)
            if roi_pixels00.sum()<LIGHT_MIN_PIXELS00:continue
            # Reading rectangular tiles is unavoidable at the boundary;
            # ONLY ROI pixels contribute to any downstream statistics.
            x=ds.read([1,2,3],window=win,masked=True)
            raw=x.filled(0)
            band_valid00=~np.any(np.ma.getmaskarray(x),axis=0)
            rgb_valid00=np.any(raw!=0,axis=0)&np.any(raw!=255,axis=0)
            rgb_valid00 &= ~((raw[0]==0)&(raw[1]==0)&(raw[2]==255))
            valid00=roi_pixels00&band_valid00&rgb_valid00
            if valid00.sum()<LIGHT_MIN_PIXELS00:continue
            luminance00=(.2126*raw[0].astype(np.float32)
                         +.7152*raw[1].astype(np.float32)
                         +.0722*raw[2].astype(np.float32))/255.
            vals00=luminance00[valid00]
            illum_rows00.append(dict(site=assign_site(p.stem),mosaic=p.name,
                row=int(row),col=int(col),roi_pixels=int(roi_pixels00.sum()),
                valid_pixels=int(valid00.sum()),
                fraction_valid_within_roi=float(valid00.sum()/roi_pixels00.sum()),
                median_luminance=float(np.median(vals00)),
                p10=float(np.quantile(vals00,.1)),p90=float(np.quantile(vals00,.9)),
                dark_pct=float(100*(vals00<.16).mean())))
            ids00=np.flatnonzero(valid00)
            take00=rng.choice(ids00,min(LIGHT_PIXELS00,len(ids00)),replace=False)
            illum_pixels00.extend((p.name,float(v)) for v in luminance00.ravel()[take00])
            accepted00+=1
        illum_attempts00.append(dict(site=assign_site(p.stem),mosaic=p.name,
            accepted_windows=accepted00,candidates_tried=tries00,
            acceptance_pct=100*accepted00/max(tries00,1),
            roi_candidate_cells=len(candidate_x),
            roi_overlap_area_m2=float(eligible_geom00.area),
            status="OK" if accepted00==LIGHT_WINDOWS00 else "LOW_SAMPLE"))
        print(f"{p.name}: {accepted00}/{LIGHT_WINDOWS00} ROI-valid windows "
              f"after {tries00} attempts",flush=True)

illum_windows00=pd.DataFrame(illum_rows00)
assert len(illum_windows00), "No valid ROI/RGB windows. Verify ROI, CRS and valid RGB bands."
attempts_df00=pd.DataFrame(illum_attempts00)
illum_summary00=illum_windows00.groupby(["site","mosaic"],as_index=False).agg(
    sampled_windows=("row","size"),median_luminance=("median_luminance","median"),
    median_p10=("median_luminance",lambda v:float(v.quantile(.10))),
    median_p90=("median_luminance",lambda v:float(v.quantile(.90))),
    mean_dark_pct=("dark_pct","mean"),
    mean_valid_fraction_within_roi=("fraction_valid_within_roi","mean"))
illum_summary00=illum_summary00.merge(attempts_df00,on=["site","mosaic"],how="outer")
display(illum_summary00.round(3))
illum_windows00.to_csv(OUT/"00_illumination_windows.csv",index=False)
illum_summary00.to_csv(OUT/"00_illumination_summary.csv",index=False)
attempts_df00.to_csv(OUT/"00_illumination_sampling_coverage.csv",index=False)

fig,axes=plt.subplots(1,2,figsize=(13,4.5),constrained_layout=True)
for name in sorted(set(k for k,_ in illum_pixels00)):
    values=[v for k,v in illum_pixels00 if k==name]
    axes[0].hist(values,bins=32,density=True,histtype="step",label=name[:25])
for name,g in illum_windows00.groupby("mosaic"):
    axes[1].scatter(g.median_luminance,g.dark_pct,s=18,alpha=.5,label=name[:25])
axes[0].set(xlabel="RGB luminance inside ROI (0–1)",ylabel="Sample density",
            title="ROI-only UAV illumination distributions")
axes[1].set(xlabel="Median ROI-window luminance",ylabel="Dark ROI pixels (%)",
            title="Within-ROI illumination heterogeneity")
for ax in axes:
    ax.grid(alpha=.15)
    ax.legend(fontsize=7)
fig.savefig(OUT/"00_illumination_comparison.png",dpi=160,bbox_inches="tight")
display(fig);plt.close(fig)
print("Only pixels INSIDE the general ROI.shp contribute to these results.")
print("V3 training AND raster inference must mask outside ROI; no predictions outside.")

blocked_mosaics00=attempts_df00.loc[
    attempts_df00.status=="NO_ROI_OVERLAP","mosaic"].tolist()
if blocked_mosaics00:
    print("BLOCKED from V3 modelling until ROI is resolved:",blocked_mosaics00)


## 06 — Exhaustive Musanga circles

Audit **`Circles_Annotations.shp`** (Musanga-positive polygons) and **`Circles_ROI.shp`** (exhaustively labelled circle boundaries). Inside circles but outside the Musanga polygons is confirmed non-Musanga; outside circles is unlabelled. Check geometry, areas, coverage and which mosaics contain the circles.


In [ ]:
# V3 Notebook 00 / Block 06 — Exhaustive Musanga circle annotations (separate supervision type)
# Confirmed names: Circles_Annotations.shp = positive Musanga polygons;
# Circles_ROI.shp = boundaries with exhaustive annotation (distinct from general ROI.shp).
from shapely.ops import unary_union
from shapely.geometry import Polygon,MultiPolygon
from shapely import make_valid

CIRCLE_LABELS=ANN_DIR/"Circles_Annotations.shp"
CIRCLE_ROIS=ANN_DIR/"Circles_ROI.shp"
print("Circle-related shapefiles available in Annotations:")
for candidate in sorted(ANN_DIR.rglob("*.shp")):
    if "circle" in candidate.name.lower():
        print(" ",candidate.relative_to(ANN_DIR))
print("Exhaustive annotation paths:")
for role,p in [("Musanga crowns",CIRCLE_LABELS),("complete-annotation boundaries",CIRCLE_ROIS)]:
    print(" ",role,":",p,"exists:",p.is_file())
    assert p.is_file(),f"Missing required exhaustive annotation input: {p}"
    for ext in (".shx",".dbf",".prj"):
        assert p.with_suffix(ext).is_file(),f"Incomplete shapefile: {p.with_suffix(ext)}"

def polygon_parts00(g):
    if isinstance(g,Polygon):return [g]
    if isinstance(g,MultiPolygon):return list(g.geoms)
    if hasattr(g,"geoms"):
        return [p for q in g.geoms for p in polygon_parts00(q)]
    return []

def inspect_and_repair00(path):
    raw=gpd.read_file(path)
    assert raw.crs is not None,f"{path.name} has no CRS"
    assert len(raw)>0 and raw.geometry.notna().all()
    bad=~raw.geometry.is_valid
    repaired=[]
    changes=[]
    for i,g in enumerate(raw.geometry):
        assert not g.is_empty,f"{path.name} empty geometry at row {i}"
        if not g.is_valid:
            parts=polygon_parts00(make_valid(g))
            assert parts,f"{path.name}: row {i} cannot be made into polygon geometry"
            fixed=unary_union(parts)
            change=abs(fixed.area-g.area)/max(fixed.area,g.area,1e-12)
            changes.append((i,change))
            assert change<=.01,(f"{path.name} row {i}: repair changes area by {change:.1%}, "
                                 "please inspect the source")
            g=fixed
        repaired.append(g)
    out=raw.copy()
    out[out.geometry.name]=repaired
    assert out.geometry.is_valid.all()
    assert out.geom_type.isin(["Polygon","MultiPolygon"]).all(),(
        f"{path.name}: expected polygon geometries")
    print(f"{path.name}: {len(raw)} features, {int(bad.sum())} invalid "
          f"(in-memory repairs only), CRS={raw.crs}")
    if changes:
        print(" Largest relative area changes:",sorted(changes,key=lambda x:-x[1])[:6])
    return out

exhaustive_rois00=inspect_and_repair00(CIRCLE_ROIS)
exhaustive_crowns00=inspect_and_repair00(CIRCLE_LABELS)
# Reproject vectors ONLY in memory to the boundary CRS; preserve source files.
crowns_aligned00=exhaustive_crowns00.to_crs(exhaustive_rois00.crs)
crown_union00=unary_union(crowns_aligned00.geometry.to_list())
# Some circle files include candidate areas that have not been exhaustively
# inventoried. Do not silently label empty candidate circles non-Musanga.
annotated_circle_ids00=[i for i,g in enumerate(exhaustive_rois00.geometry)
                        if crown_union00.intersection(g).area>1.]
assert annotated_circle_ids00,"No circle boundaries overlap positive polygons"
unverified_circle_ids00=[i for i in range(len(exhaustive_rois00))
                         if i not in annotated_circle_ids00]
print("Circles with matching exhaustive polygons:",annotated_circle_ids00)
if unverified_circle_ids00:
    print("Other circles NOT assumed exhaustive:",unverified_circle_ids00)
circle_union00=unary_union([exhaustive_rois00.geometry.iloc[i]
                            for i in annotated_circle_ids00])
contained00=crown_union00.intersection(circle_union00)
assert contained00.area>0,"Exhaustive Musanga polygons do not intersect Circles_ROI boundaries"
coverage00=contained00.area/crown_union00.area
print(f"Annotated Musanga area inside exhaustive ROI union: {coverage00:.2%}")
print("Musanga polygon area outside circles (ignored):",
      round(crown_union00.area-contained00.area,2),"CRS area units squared")
if coverage00<.85:
    warnings.warn("Less than 85% of annotated Musanga polygon area falls inside circles; "
                  "verify these are matching ROI and crown files.")

circle_rows00=[]
for i,geom in enumerate(exhaustive_rois00.geometry):
    pos=contained00.intersection(geom)
    circle_rows00.append(dict(circle_row=i,has_exhaustive_positives=i in annotated_circle_ids00,
        circle_area_units2=float(geom.area),
        musanga_area_units2=float(pos.area),
        fraction_musanga=float(pos.area/geom.area) if geom.area else np.nan,
        n_crowns_intersecting=int(crowns_aligned00.geometry.intersects(geom).sum())))
circle_summary00=pd.DataFrame(circle_rows00)
display(circle_summary00)
circle_summary00.to_csv(OUT/"00_exhaustive_circle_area_audit.csv",index=False)

# Determine which mosaic(s) overlap each exhaustive circle; no assumption
# that this dataset is Yangambi-only or that all mosaic CRS are identical.
relations00=[]
for circle_idx in annotated_circle_ids00:
    g=exhaustive_rois00.geometry.iloc[circle_idx]
    for image_path in rgb_paths:
        with rasterio.open(image_path) as src:
            geom_reprojected=gpd.GeoSeries([g],crs=exhaustive_rois00.crs).to_crs(src.crs).iloc[0]
            footprint=box(*src.bounds)
            intersection=float(geom_reprojected.intersection(footprint).area)
            if intersection>0:
                relations00.append(dict(circle_row=circle_idx,
                    mosaic=image_path.name,site=assign_site(image_path.stem),
                    intersecting_area_mosaic_crs_units2=intersection,
                    circle_fraction_in_mosaic_bbox=intersection/max(geom_reprojected.area,1e-12)))
relations_df00=pd.DataFrame(relations00)
print("Exhaustive circle ↔ RGB mosaic overlaps:")
display(relations_df00)
relations_df00.to_csv(OUT/"00_exhaustive_circle_mosaic_links.csv",index=False)
if len(relations00)==0:
    warnings.warn("No exhaustive circles overlap any RGB mosaic; investigate CRS and georeferencing.")

print("Supervision contract: Musanga pixels are positive INSIDE the circles;")
print("unannotated pixels are confirmed non-Musanga ONLY INSIDE the circles.")
print("Outside circles: UNLABELLED (not negative).")


## 07 — Crown annotation inventory: partial versus exhaustive

Class and polygon-count review from V2 Notebook 00: Musanga versus other labelled crowns, original class names, polygon area distribution, and counts above/below the historical 5 m² rule. **Do not confuse partial-crown absence with non-Musanga.**


In [ ]:
# Block 07 — Individual tree-crown annotation overview, separate from exhaustive circles
# Same Class_ID=1 convention and ≥5m² compatibility statistic used in V2 NB00.
MUSANGA_CLASS00=1
MIN_CROWN_AREA00=5.
TREE_SOURCES00={}
annotation_rows00=[];annotation_labels00=[]
for site,p in ann_site.items():
    g=inspect_and_repair00(p)
    assert g.crs.is_projected,f"Tree annotations {p.name} require projected CRS for area metrics"
    area=g.geometry.area
    if "Class_ID" in g.columns:
        cls=pd.to_numeric(g["Class_ID"],errors="coerce")
        pos=cls.eq(MUSANGA_CLASS00)&cls.notna()
        known=cls.notna()
        field="Class_ID"
    elif "Espece" in g.columns:
        cls=g["Espece"].fillna("").astype(str)
        pos=cls.str.casefold().str.contains("musanga")
        known=cls.str.len()>0
        field="Espece"
        print("NOTE:",p.name,"Musanga inferred by 'musanga' substring in Espece; inspect raw labels.")
    else:
        cls=pd.Series(["<unknown>"]*len(g),index=g.index)
        pos=pd.Series(False,index=g.index);known=pos
        field="UNMAPPED"
        warnings.warn(f"{p.name} has no Class_ID or Espece; Musanga comparison cannot be trusted")
    g=g.copy();g["_musanga00"]=pos.astype(bool);g["_known00"]=known.astype(bool);g["_area_m2_00"]=area
    TREE_SOURCES00[site]=g
    annotation_rows00.append(dict(annotation_type="partial_tree_crowns",site=site,
        file=p.name,n_polygons=len(g),label_field=field,n_musanga=int(pos.sum()),
        n_other_known=int((known&~pos).sum()),n_unknown=int((~known).sum()),
        n_below_5m2=int((area<MIN_CROWN_AREA00).sum()),
        n_musanga_ge5m2=int((pos&(area>=MIN_CROWN_AREA00)).sum()),
        total_musanga_area_m2=float(area[pos].sum()),
        median_musanga_crown_m2=float(area[pos].median())))
    for label,count in cls.fillna("<missing>").astype(str).value_counts().items():
        annotation_labels00.append(dict(site=site,file=p.name,
                                        raw_label=label,count=int(count)))
exhaustive_area00=crowns_aligned00.geometry.area
annotation_rows00.append(dict(annotation_type="exhaustive_musanga_polygons",site="ALL_CIRCLES",
    file=CIRCLE_LABELS.name,n_polygons=len(exhaustive_crowns00),
    label_field="all polygons Musanga",n_musanga=len(exhaustive_crowns00),
    n_other_known=0,n_unknown=0,n_below_5m2=int((exhaustive_area00<5).sum()),
    n_musanga_ge5m2=int((exhaustive_area00>=5).sum()),
    total_musanga_area_m2=float(exhaustive_area00.sum()),
    median_musanga_crown_m2=float(exhaustive_area00.median())))
annotation_overview00=pd.DataFrame(annotation_rows00)
display(annotation_overview00.round(2))
display(pd.DataFrame(annotation_labels00))
annotation_overview00.to_csv(OUT/"00_annotation_crown_overview.csv",index=False)
pd.DataFrame(annotation_labels00).to_csv(OUT/"00_partial_crown_class_counts.csv",index=False)
print("IMPORTANT: blank outside partially annotated tree crowns is UNKNOWN;")
print("blank inside exhaustive Circles_ROI.shp but outside Musanga polygons is NON-Musanga.")


## 08 — Consistency between partial tree crowns and exhaustive circle annotations

Following the exact-vector comparison in `Crown/02_SAM1_Spatial_Error_Diagnostics.ipynb`, compare tree-annotated **Musanga only** with exhaustively annotated Musanga, within each circle. Report precision, recall, area F1/Dice, IoU, overlap and cover fractions. Explicit non-Musanga tree labels conflicting with the exhaustive positives are counted separately. **These values measure annotation-footprint reproducibility**, not true-negative or complete model performance.

A second, **object-level crown matching** score uses one-to-one polygon matches at IoU ≥ 0.5; both object F1 and area-weighted Dice/F1 are reported, because they answer different questions.


In [ ]:
# Block 08 — Area-based agreement of tree vs exhaustive Musanga polygons, each circle
# Adapt Crown/02 canopy comparison: both, tree-only, exhaustive-only.
# Partial source is NOT assumed exhaustive. Metrics are footprint reproducibility.
from shapely.ops import unary_union
from shapely.geometry import Polygon
from shapely.strtree import STRtree
assert exhaustive_rois00.crs.is_projected, "Circle ROI CRS must be projected for m²"
def union_clipped00(geoms,roi):
    parts=[]
    for geom in geoms:
        if geom is not None and not geom.is_empty and geom.intersects(roi):
            intersect=geom.intersection(roi)
            if not intersect.is_empty and intersect.area>0:parts.append(intersect)
    return unary_union(parts) if parts else Polygon()
circle_comparisons00=[]
circle_footprints00={}
# Preproject partial crowns once into circle CRS, and separate known non-Musanga crowns.
partial_by_site00={}
negative_by_site00={}
for site,g in TREE_SOURCES00.items():
    aligned=g.to_crs(exhaustive_rois00.crs)
    partial_by_site00[site]=aligned.loc[aligned["_musanga00"]].copy()
    negative_by_site00[site]=aligned.loc[aligned["_known00"]&~aligned["_musanga00"]].copy()

for circle_id00 in annotated_circle_ids00:
    roi=exhaustive_rois00.geometry.iloc[circle_id00]
    matches=relations_df00.loc[relations_df00.circle_row==circle_id00].copy()
    assert len(matches),f"Circle {circle_id00} does not overlap any RGB mosaic"
    matches=matches.sort_values("circle_fraction_in_mosaic_bbox",ascending=False)
    # Choose site associated with best RGB footprint overlap; multiple MOS images
    # share a single source annotation file, so choose the best one for visualization.
    chosen00=matches.iloc[0]
    site00=chosen00.site
    mosaic00=chosen00.mosaic
    tree_g00=partial_by_site00[site00]
    negatives00=negative_by_site00[site00]
    ref00=contained00.intersection(roi)
    tree00=union_clipped00(tree_g00.geometry,roi)
    both00=ref00.intersection(tree00)
    tree_only00=tree00.difference(ref00)
    circle_only00=ref00.difference(tree00)
    a_ref00=float(ref00.area);a_tree00=float(tree00.area)
    a_both00=float(both00.area)
    denom00=a_ref00+a_tree00-a_both00
    precision00=a_both00/a_tree00 if a_tree00>0 else np.nan
    recall00=a_both00/a_ref00 if a_ref00>0 else np.nan
    f100=2*a_both00/(a_ref00+a_tree00) if a_ref00+a_tree00>0 else np.nan
    iou00=a_both00/denom00 if denom00>0 else np.nan
    n_exh00=int(crowns_aligned00.geometry.intersects(roi).sum())
    n_tree00=int(tree_g00.geometry.intersects(roi).sum())
    n_known_negative00=int(negatives00.geometry.intersects(roi).sum())
    # Control: area where an explicitly non-Musanga tree polygon contradicts
    # an exhaustively annotated Musanga crown (possible label conflict).
    negpoly00=union_clipped00(negatives00.geometry,roi)
    conflict_area00=float(ref00.intersection(negpoly00).area)
    # Object-level control: one-to-one crown matches with polygon IoU >= 0.5.
    # Separate from area-weighted F1/Dice above; unmatched crowns can simply
    # reflect that the partial inventory was not designed to be exhaustive.
    ref_crowns00=[g.intersection(roi) for g in crowns_aligned00.geometry
                  if g.intersects(roi) and g.intersection(roi).area>0]
    tree_crowns00=[g.intersection(roi) for g in tree_g00.geometry
                   if g.intersects(roi) and g.intersection(roi).area>0]
    pair_scores00=[]
    for ti,tg in enumerate(tree_crowns00):
        for ri,rg in enumerate(ref_crowns00):
            if not tg.intersects(rg):continue
            ov=float(tg.intersection(rg).area)
            pair_iou=ov/max(float(tg.area+rg.area-ov),1e-12)
            if pair_iou>=.5:pair_scores00.append((pair_iou,ti,ri))
    used_tree00=set();used_ref00=set()
    for iou_pair,ti,ri in sorted(pair_scores00,reverse=True):
        if ti in used_tree00 or ri in used_ref00:continue
        used_tree00.add(ti);used_ref00.add(ri)
    nmatched00=len(used_ref00)
    object_precision00=nmatched00/len(tree_crowns00) if tree_crowns00 else np.nan
    object_recall00=nmatched00/len(ref_crowns00) if ref_crowns00 else np.nan
    object_f100=2*nmatched00/(len(tree_crowns00)+len(ref_crowns00)) if (
        tree_crowns00 or ref_crowns00) else np.nan
    record00=dict(circle_row=circle_id00,site=site00,mosaic=mosaic00,
        object_matches_iou_ge_05=nmatched00,
        object_tree_crowns=len(tree_crowns00),object_exhaustive_crowns=len(ref_crowns00),
        object_precision=object_precision00,object_recall=object_recall00,
        object_F1=object_f100,
        roi_m2=float(roi.area),reference_musanga_m2=a_ref00,
        partial_tree_musanga_m2=a_tree00,both_m2=a_both00,
        tree_only_m2=float(tree_only00.area),exhaustive_only_m2=float(circle_only00.area),
        exhaustive_cover_pct=100*a_ref00/roi.area,
        partial_tree_cover_pct=100*a_tree00/roi.area,
        cover_difference_pp=100*(a_tree00-a_ref00)/roi.area,
        area_precision=precision00,area_recall=recall00,area_F1_Dice=f100,
        area_IoU=iou00,n_exhaustive_polygons_touching=n_exh00,
        n_partial_musanga_crowns_touching=n_tree00,
        n_partial_known_nonmusanga_crowns_touching=n_known_negative00,
        exhaustive_Musanga_vs_known_nonmusanga_conflict_m2=conflict_area00,
        mosaic_bbox_fraction=float(chosen00.circle_fraction_in_mosaic_bbox))
    circle_comparisons00.append(record00)
    circle_footprints00[circle_id00]=dict(roi=roi,reference=ref00,tree=tree00,
                                          both=both00,tree_only=tree_only00,
                                          exhaustive_only=circle_only00,site=site00,
                                          mosaic=mosaic00)
circle_comparison00=pd.DataFrame(circle_comparisons00)
display(circle_comparison00.round(3))
circle_comparison00.to_csv(OUT/"00_partial_vs_exhaustive_per_circle.csv",index=False)
summary00=circle_comparison00.groupby("site",as_index=False).agg(
    circles=("circle_row","size"),reference_area_m2=("reference_musanga_m2","sum"),
    partial_area_m2=("partial_tree_musanga_m2","sum"),both_m2=("both_m2","sum"),
    roi_m2=("roi_m2","sum"),conflict_m2=("exhaustive_Musanga_vs_known_nonmusanga_conflict_m2","sum"))
summary00["area_precision"]=summary00.both_m2/summary00.partial_area_m2.replace(0,np.nan)
summary00["area_recall"]=summary00.both_m2/summary00.reference_area_m2.replace(0,np.nan)
summary00["area_F1_Dice"]=2*summary00.both_m2/(summary00.reference_area_m2+summary00.partial_area_m2).replace(0,np.nan)
summary00["area_IoU"]=summary00.both_m2/(summary00.reference_area_m2+summary00.partial_area_m2-summary00.both_m2).replace(0,np.nan)
summary00["fraction_exhaustive_pct"]=100*summary00.reference_area_m2/summary00.roi_m2
summary00["fraction_partial_pct"]=100*summary00.partial_area_m2/summary00.roi_m2
print("Per-site pooled footprint agreement (not independent-pixel classification):")
display(summary00.round(3))
summary00.to_csv(OUT/"00_partial_vs_exhaustive_by_site.csv",index=False)
print("Interpretation: a missed exhaustive crown may reflect missing partial annotation,")
print("rather than inability of an annotator/model to recognize the crown.")
print("Known-negative conflicts are listed separately from unannotated background.")


## 09 — Per-circle RGB and annotation mismatch figures

For each exhaustive circle, show RGB alongside the Crown/SAM categorical overlap style: green = both sources mark Musanga, orange = tree-only, blue = exhaustive-only, grey = neither. Figures are downsampled to protect memory; reported metrics use exact polygon geometry.


In [ ]:
# Block 09 — Crown/SAM-style per-circle RGB and categorical agreement maps
from rasterio.windows import from_bounds,Window
from rasterio.enums import Resampling
from rasterio.features import rasterize
from affine import Affine
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

MAX_PREVIEW_DIM00=1100  # downsampling affects the display only, NEVER vector metrics.
def geom_mask_preview00(geom,shape,transform):
    if geom.is_empty:return np.zeros(shape,dtype=bool)
    return rasterize([(geom,1)],out_shape=shape,transform=transform,fill=0,
                     dtype=np.uint8).astype(bool)

def circle_map00(circle_id):
    fp=circle_footprints00[circle_id]
    path=next(p for p in rgb_paths if p.name==fp["mosaic"])
    with rasterio.open(path) as ds:
        # All geometry operations here use the image CRS, rather than assuming
        # all forest sites share a particular EPSG code.
        trans_geom=lambda g:gpd.GeoSeries([g],crs=exhaustive_rois00.crs).to_crs(ds.crs).iloc[0]
        region=trans_geom(fp["roi"])
        r=region.bounds
        raw=from_bounds(*r,transform=ds.transform)
        raw=raw.round_offsets().round_lengths()
        win=raw.intersection(Window(0,0,ds.width,ds.height))
        ww=int(win.width);hh=int(win.height)
        assert ww>0 and hh>0,f"Circle {circle_id} outside RGB raster"
        scale=min(1.,MAX_PREVIEW_DIM00/max(ww,hh))
        out_w=max(1,int(round(ww*scale)));out_h=max(1,int(round(hh*scale)))
        image=ds.read([1,2,3],window=win,out_shape=(3,out_h,out_w),
                      resampling=Resampling.bilinear).transpose(1,2,0).astype(np.float32)
        transform=ds.window_transform(win)*Affine.scale(ww/out_w,hh/out_h)
        bounds=ds.window_bounds(win)
        extent=(bounds[0],bounds[2],bounds[1],bounds[3])
    roi_mask=geom_mask_preview00(region,(out_h,out_w),transform)
    reference=geom_mask_preview00(trans_geom(fp["reference"]),(out_h,out_w),transform)
    tree=geom_mask_preview00(trans_geom(fp["tree"]),(out_h,out_w),transform)
    # Classes match Crown/02: 0 neither, 1 both, 2 tree-only, 3 exhaustive-only, 4 outside ROI.
    classes=np.zeros((out_h,out_w),dtype=np.uint8)
    classes[roi_mask & reference & tree]=1
    classes[roi_mask & ~reference & tree]=2
    classes[roi_mask & reference & ~tree]=3
    classes[~roi_mask]=4
    pixels=image.reshape(-1,3)
    valid=np.any(pixels!=0,axis=1)&np.any(pixels!=255,axis=1)
    lo,hi=np.percentile(pixels[valid] if valid.any() else pixels,[2,98])
    display_rgb=np.clip((image-lo)/max(hi-lo,1.),0,1)
    cmap=ListedColormap(["#d1d1d1","#3e9060","#ea884e","#4e83cc","#ffffff"])
    fig,axs=plt.subplots(1,2,figsize=(12.5,5.6),constrained_layout=True)
    axs[0].imshow(display_rgb,extent=extent,origin="upper")
    axs[1].imshow(classes,extent=extent,origin="upper",
                  cmap=cmap,vmin=-.5,vmax=4.5,interpolation="nearest")
    for ax in axs:
        bd=region.boundary
        for line in (bd.geoms if hasattr(bd,"geoms") else [bd]):
            if hasattr(line,"xy"):
                x,y=line.xy
                ax.plot(x,y,color="black",linewidth=1.0)
        ax.set_aspect("equal");ax.set_xticks([]);ax.set_yticks([])
    row=circle_comparison00.loc[circle_comparison00.circle_row==circle_id].iloc[0]
    axs[0].set_title(f"{fp['site']} circle {circle_id}: native RGB preview")
    axs[1].set_title(f"Tree vs exhaustive Musanga | F1={row.area_F1_Dice:.3f}, IoU={row.area_IoU:.3f}")
    legend=[Patch(facecolor=cmap.colors[i],label=label) for i,label in
        [(0,"Neither"),(1,"Both Musanga"),(2,"Tree-only"),(3,"Exhaustive-only")]]
    axs[1].legend(handles=legend,loc="lower left",fontsize=8)
    fig.savefig(OUT/f"00_circle_{circle_id:03d}_annotation_agreement.png",
                dpi=175,bbox_inches="tight")
    display(fig);plt.close(fig)
for circle_id in circle_footprints00:
    print(f"Drawing circle {circle_id+1}/{len(circle_footprints00)} "
          f"({circle_footprints00[circle_id]['site']})",flush=True)
    circle_map00(circle_id)
print("All per-circle maps saved under:",OUT)


## 10 — Save the V3 data contract

Save reproducible source paths, spatial QA tables and the exhaustive-annotation supervision rules for the next V3 notebooks.


In [ ]:
# V3 Notebook 00 / Block 10 — Persistent input manifest and next-step checks
manifest=dict(schema_version="v3_input_audit_3",
    data_root=str(DATA),output_root=str(OUT),
    site_groups=["Baego","MOS","Yangambi","Yoko"],
    mosaics=[dict(site=assign_site(p.stem),file=p.name,path=str(p),
                  bytes=p.stat().st_size) for p in rgb_paths],
    shared_annotations={site:str(path) for site,path in ann_site.items()},
    roi=str(ROI_FILE),
    modelling_mask_policy="ROI.shp is mandatory for input sampling, training and prediction; outside = ignored/no prediction",
    exhaustive_mask_policy="Circles_ROI.shp only defines exhaustive annotation support",
    exhaustive_annotations=dict(positive_polygons=str(CIRCLE_LABELS),
        complete_roi_boundaries=str(CIRCLE_ROIS),
        circle_count=int(len(exhaustive_rois00)),
        musanga_polygons=int(len(exhaustive_crowns00)),
        polygon_area_containment_fraction=float(coverage00),
        supervision_rule="outside Musanga polygons = negative ONLY within Circles_ROI; outside ROI = unknown"),
    dsms=[str(p) for p in dsm_paths],
    illumination_summary_csv=str(OUT/"00_illumination_summary.csv"),
    crown_overview_csv=str(OUT/"00_annotation_crown_overview.csv"),
    circle_consistency_csv=str(OUT/"00_partial_vs_exhaustive_per_circle.csv"),
    circle_consistency_by_site_csv=str(OUT/"00_partial_vs_exhaustive_by_site.csv"),
    exclusions=["Luki"],
    remarks=[
       "Six orthomosaics across four site groups; three MOS mosaics use one annotation dataset.",
       "All inputs remain untouched; this notebook inventories files, CRS, label schema and spatial overlap.",
       "ROI modelling extent has been checked only at footprint level; clipping/pixel mask follows in V3/01.",
       "Musanga class coding requires deliberate confirmation prior to model training."
    ])
(OUT/"00_input_manifest.json").write_text(json.dumps(manifest,indent=2))
print("V3 Notebook 00 complete. Manifest:",OUT/"00_input_manifest.json")
print("RGB mosaics:",len(rgb_paths),"| sites:",sorted(set(assign_site(p.stem) for p in rgb_paths)))
print("DSM:",[p.name for p in dsm_paths],"| ROI:",ROI_FILE.name)
print("Next: V3/01 reference polygons and ROI masks; then U-Net development.")
